# Day 1: A keyword-based RAG reference notebook

## Introduction
Build an Insurellm question-answering assistant using **retrieval-augmented generation
(RAG)**: load local documents, retrieve matching facts, add them to a prompt, and
ask a language model to answer. This first version uses exact keywords, not
embeddings or a vector database.

**Pipeline:** Markdown files -> separate employee/product dictionaries -> labeled
matches -> system prompt + conversation history -> OpenAI -> Gradio.

### How to use this reference
- Select the project's Python environment and run cells from top to bottom.
- Start the kernel inside this repository; document paths are resolved from the
  current working directory and its parents.
- Put `OPENAI_API_KEY` in your local `.env`, never in the notebook. The configured
  model must be available to your API account.
- Loading, retrieval, and the tests are local. Sending a question through the UI
  makes a paid API request and sends the selected context and history to OpenAI.
  Use the course's example documents, not confidential employee information.
- For offline study without credentials, skip section 2 and the UI launch;
  the remaining examples and tests do not require an API connection.

Stored outputs are cleared so old results do not look like results from a new run.

## 1. Imports
`Path` handles local paths, dotenv reads configuration, OpenAI provides the API
client, and Gradio provides the chat interface. No API requests are made here.

In [ ]:
# Use pathlib for file discovery; a separate glob import is not needed.
import os
from pathlib import Path

import gradio as gr
from dotenv import load_dotenv
from openai import OpenAI

## 2. Configure the API client
The local `.env` takes precedence over existing environment variables, matching
the course setup. Missing credentials stop this cell with an actionable error.
Creating the client does not send a completion request.

The default model below preserves this notebook's original choice. Model access
and support for `reasoning_effort="none"` must be checked with your API provider;
the offline tests do not certify either.

In [ ]:
# Keep credentials outside the notebook and never print their value.
load_dotenv(override=True)
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("Set OPENAI_API_KEY in your local .env before using live chat.")

MODEL = "gpt-6-luna"
openai = OpenAI()
print("OpenAI client configured; no API request has been sent.")

## 3. Load employee documents
Search upward for `week5/knowledge-base/employees` so this works from the
repository root, this notebook's directory, or the course's `week5` directory.
Missing directories and empty directories raise errors rather than silently
producing an empty dictionary.

Each key is a normalized surname; each value is document text. Apostrophes are
removed from lookup keys (`O'Brien` -> `obrien`) to match the question tokenizer.
**Shared surnames retain all matching documents**, separated by a divider.

In [ ]:
# Notebook kernels resolve relative paths from Path.cwd(), not the notebook file.
current_dir = Path.cwd()
for directory in (current_dir, *current_dir.parents):
    employees_dir = directory / "week5" / "knowledge-base" / "employees"
    if employees_dir.is_dir():
        break
else:
    raise FileNotFoundError(
        f"Could not find week5/knowledge-base/employees from {current_dir}. "
        "Start the notebook kernel inside the project directory."
    )

employee_files = sorted(employees_dir.glob("*.md"))
if not employee_files:
    raise FileNotFoundError(f"No employee Markdown files found in {employees_dir}")

employees = {}
for filename in employee_files:
    surname = filename.stem.split()[-1].lower()
    name = "".join(ch for ch in surname if ch.isalpha())
    content = filename.read_text(encoding="utf-8")
    # A surname is not unique: keep both records instead of overwriting one.
    if name in employees:
        employees[name] += "\n\n---\n\n" + content
    else:
        employees[name] = content

print(f"Loaded {len(employee_files)} employee files into {len(employees)} surname keys.")

In [ ]:
# Inspect keys without printing the entire employee knowledge base.
sorted(employees)

In [ ]:
# Direct lookup confirms the file was read before testing retrieval.
employees["lancaster"]

**Employee-loading summary:** `employees` contains UTF-8 document text indexed
by surname. Several files may share a key. Retrieval by surname alone cannot
disambiguate those people; their original document headings remain in the text.

## 4. Load product documents independently
Products live in their own dictionary. Resetting `products` does not erase
`employees`, which prevents the earlier empty-result problem.

In [ ]:
# Resolve the product folder independently so path errors are explicit.
current_dir = Path.cwd()
for directory in (current_dir, *current_dir.parents):
    products_dir = directory / "week5" / "knowledge-base" / "products"
    if products_dir.is_dir():
        break
else:
    raise FileNotFoundError(
        f"Could not find week5/knowledge-base/products from {current_dir}. "
        "Start the notebook kernel inside the project directory."
    )

product_files = sorted(products_dir.glob("*.md"))
if not product_files:
    raise FileNotFoundError(f"No product Markdown files found in {products_dir}")

products = {}
for filename in product_files:
    # Product names use the entire filename stem, rather than a person's surname.
    name = filename.stem.lower()
    products[name] = filename.read_text(encoding="utf-8")

print(f"Loaded {len(product_files)} product files from {products_dir}")

In [ ]:
# Confirm product keys and employee keys coexist after both loaders run.
sorted(products), "lancaster" in employees

**Loading summary:** employee and product documents are separate sources. Rerunning
either loader rebuilds only its own dictionary. The current example product names
are single words; multiword names would require phrase matching.

## 5. Retrieve matching context
This is the **retrieval** step. Normalize case, remove punctuation, and search
each distinct word in both dictionaries. Labels identify each source.
Repeated keywords return a record only once, while query order is preserved.

This is exact keyword matching: `Lancaster?` works, but `Avery`, synonyms, pronouns
such as "her", and meaning-based questions are not automatically resolved.
Punctuation is removed rather than replaced, so hyphenated combinations may
merge into a single token.

In [ ]:
# Instructions are separate from retrieved facts and conversation history.
SYSTEM_PREFIX = """
You represent Insurellm, the Insurance Tech company.
You are an expert in answering questions about Insurellm, its employees and its products.
You are provided with additional context that might be relevant to the user's question.
Give brief, accurate answers. If you don't know the answer, say so.

Relevant context:
"""

In [ ]:
def get_relevant_context(message):
    # Use the same apostrophe-removal rule as the employee lookup keys.
    text = "".join(ch for ch in message if ch.isalpha() or ch.isspace())
    words = dict.fromkeys(text.lower().split())
    sources = [("employee", employees), ("product", products)]

    return [
        f"[{category}: {word}]\n{entries[word]}"
        for word in words
        for category, entries in sources
        if word in entries
    ]

In [ ]:
# One surname produces one labeled employee-context string.
get_relevant_context("Who is Lancaster?")

In [ ]:
# A question can retrieve from both sources without switching functions.
get_relevant_context("Who is Lancaster and what is Carllm?")

**Retrieval summary:** the result is a list of strings, not a model-generated
answer. An empty list means no query token matched a loaded key. Check the
dictionary keys and rerun the loaders before blaming the model.

## 6. Format context for the prompt
Join the retrieved strings into a single block. An unmatched query gets an
explicit no-context message instead of invented supporting facts.

In [ ]:
def additional_context(message):
    relevant_context = get_relevant_context(message)
    if not relevant_context:
        return "There is no relevant context available."

    # Blank lines keep separate document excerpts readable in the prompt.
    return "The following additional context is available:\n\n" + "\n\n".join(relevant_context)

In [ ]:
# Use the actual document name; surname retrieval does not verify a first name.
print(additional_context("Who is Avery Lancaster?"))

## 7. Build the chat request
This is the **augmentation and generation** step: prepend instructions and
retrieved facts, preserve Gradio's message-format history, then append the current
user message. The API field is `content`, not `context`.

The function below sends a live request only when called. It retrieves using the
current question, not conversation history. A follow-up like "What is her role?"
may therefore miss relevant documents even though earlier turns are retained.
API errors are allowed to surface instead of being disguised as successful replies.

In [ ]:
def chat(message, history):
    system_message = SYSTEM_PREFIX + additional_context(message)
    # With type="messages", history is already a list of role/content dictionaries.
    messages = [
        {"role": "system", "content": system_message}
    ] + history + [
        {"role": "user", "content": message}
    ]
    response = openai.chat.completions.create(
        model=MODEL,
        reasoning_effort="none",
        messages=messages,
    )
    answer = response.choices[0].message.content
    if answer is None:
        raise ValueError("The model returned no text content; inspect the API response.")
    return answer

## 8. Offline regression tests
These assertions use the actual local documents. They check loading, case and
punctuation handling, both categories, repeated keywords, apostrophes, shared
surnames, and no-match behavior.

The chat tests replace the API client temporarily with a mock, so they cost
nothing, work without credentials, and verify request construction rather than
model quality. A passing test does **not** prove live model access or API
parameter compatibility.

In [ ]:
# Every input document must survive loading, including shared surnames.
for filename in employee_files:
    surname = filename.stem.split()[-1].lower()
    key = "".join(ch for ch in surname if ch.isalpha())
    assert filename.read_text(encoding="utf-8") in employees[key], filename.name
for filename in product_files:
    assert products[filename.stem.lower()] == filename.read_text(encoding="utf-8")

employee_result = get_relevant_context("Who is LANCASTER?")
assert employee_result == [f"[employee: lancaster]\n{employees['lancaster']}"]
product_result = get_relevant_context("What is Carllm?")
assert product_result == [f"[product: carllm]\n{products['carllm']}"]
assert get_relevant_context("Lancaster and Carllm?") == employee_result + product_result
assert get_relevant_context("Lancaster Lancaster?") == employee_result
assert get_relevant_context("Who is O'Brien?") == [f"[employee: obrien]\n{employees['obrien']}"]
assert get_relevant_context("unmatchedkeyword") == []
assert get_relevant_context("") == []
assert additional_context("unmatchedkeyword") == "There is no relevant context available."
assert employees["lancaster"] in additional_context("Lancaster?")
print("Offline document-loading, retrieval, and context tests passed.")

In [ ]:
from types import SimpleNamespace
from unittest.mock import Mock, patch


def test_chat_request():
    fake_response = SimpleNamespace(
        choices=[SimpleNamespace(message=SimpleNamespace(content="Test answer"))]
    )
    create = Mock(return_value=fake_response)
    fake_client = SimpleNamespace(
        chat=SimpleNamespace(completions=SimpleNamespace(create=create))
    )
    history = [
        {"role": "user", "content": "Earlier question"},
        {"role": "assistant", "content": "Earlier answer"},
    ]
    original_history = [entry.copy() for entry in history]

    # Restore the real client/model after the block, even if an assertion fails.
    with patch.dict(chat.__globals__, {"openai": fake_client, "MODEL": "offline-test-model"}):
        for previous_turns in ([], history):
            assert chat("Who is Lancaster?", previous_turns) == "Test answer"
            request = create.call_args.kwargs
            assert request["model"] == "offline-test-model"
            assert request["reasoning_effort"] == "none"
            assert request["messages"] == [
                {"role": "system", "content": SYSTEM_PREFIX + additional_context("Who is Lancaster?")},
                *previous_turns,
                {"role": "user", "content": "Who is Lancaster?"},
            ]
        assert history == original_history

        fake_response.choices[0].message.content = None
        try:
            chat("Lancaster?", [])
        except ValueError as error:
            assert "no text content" in str(error)
        else:
            raise AssertionError("A missing response must raise an explicit error.")


test_chat_request()
print("Offline chat-request and missing-response tests passed; no API calls were made.")

**Testing summary:** a successful run prints two passing-test messages. A failure
identifies an assumption that no longer holds; do not remove an assertion just to
make the notebook pass. These are sample-data regression tests, so update fixture
names deliberately if you replace the course documents.

## 9. Launch the interactive application (optional, live)
Run this only after configuring the client in section 2. Gradio opens a local
interface; each submitted message calls `chat` and may incur API charges.
No public sharing link is requested. The `view` variable retains the interface so
you can close it with `view.close()`. Close it before rerunning this launch cell.

In [ ]:
# Keep the interface object instead of assigning launch()'s return tuple to view.
view = gr.ChatInterface(chat, type="messages")
view.launch(inbrowser=True)

## Conclusion
This notebook implements a complete, small keyword-based RAG pipeline:
1. Load UTF-8 Markdown files with explicit path and empty-directory checks.
2. Keep employees and products separate, preserving shared-surname documents.
3. Retrieve case-insensitive keyword matches and label their sources.
4. Format context, preserve chat history, and send valid `role`/`content` messages.
5. Test local logic and request construction before using the live UI.

### Troubleshooting reference
| Symptom | Check |
|---|---|
| `FileNotFoundError` | Start the kernel inside the repository; confirm `week5/knowledge-base` exists. |
| `KeyError` or `[]` for Lancaster | Rerun the employee loader and inspect `employees.keys()`; do not reset it in the product cell. |
| API says `content` is null | Use the `content` field, not `context`, for system and user messages. |
| UI still behaves like old code | Rerun the edited function cells, close the old interface, and relaunch. |
| Authentication/model/parameter error | Check your API key, account model access, and supported request parameters. |
| A follow-up retrieves nothing | Retrieval only sees the current message; repeat a surname or product keyword. |

### Limits and next steps
This is a teaching baseline, not a production HR assistant. Exact-word matching
cannot understand semantic similarity, first-name identity, or conversational
references. Shared surnames retrieve multiple people, multiword products are not
supported, and large retrieved documents/history can exceed a model's context
limit. Source text is not a trusted instruction channel, and live answer
correctness is not guaranteed by offline tests.

Later improvements include chunking documents, attaching file-level citations,
embedding-based retrieval, conversation-aware queries, context-size limits, and
evaluating answer faithfulness. For now, keep the simple pipeline visible and use
the tests to distinguish loading, retrieval, prompt, and API problems.